In [2]:
import json
from pathlib import Path

DATA_DIR = Path("data")

with open(DATA_DIR / "empiar_11759" / "metadata.json", "r") as f:
    empiar_metadata = json.load(f)

with open(DATA_DIR / "hemibrain" / "metadata.json", "r") as f:
    hemibrain_metadata = json.load(f)

with open(DATA_DIR / "idr" / "metadata.json", "r") as f:
    idr_metadata = json.load(f)

print("Loaded metadata files:")
print("- EMPIAR")
print("- Hemibrain")
print("- IDR")

Loaded metadata files:
- EMPIAR
- Hemibrain
- IDR


In [3]:
epfl_metadata =   {
      "name": "EPFL Electron Microscopy Dataset",
      "source": "EPFL",
      "organism": "unknown (brain tissue)",
      "brain_region": "CA1 hippocampus",
      "modality": "Electron Microscopy",
      "volume_size_voxels": [1065, 2048, 1536],
      "voxel_resolution_nm": [5, 5, 5],
      "physical_volume_um": [5, 5, 5],
      "file_format": "multipage TIFF",
      "annotation": {
        "annotated_structures": ["mitochondria"],
        "other_structures_of_interest": ["synapses", "vesicles", "cell boundaries"],
        "subvolumes": {
          "count": 2,
          "slices_per_subvolume": 165,
          "usage": {
            "top_part": "training",
            "bottom_part": "testing"
          }
        }
      },
      "data_splits": [
        "full dataset",
        "training sub-volume",
        "testing sub-volume",
        "ground truth training sub-volume",
        "ground truth testing sub-volume"
      ],
      "acquisition": {
        "contributors": [
          "Graham Knott",
          "Marco Cantoni"
        ],
        "institution": "EPFL"
      },
      "intended_use": [
        "mitochondria segmentation",
        "synapse segmentation",
        "general neuroscience research"
      ],
      "related_publication": {
        "title": "Learning for Structured Prediction Using Approximate Subgradient Descent with Working Sets",
        "venue": "CVPR 2013"
      },
      "additional_outputs": [
        "binary cube results"
      ]
    }

In [4]:
openorganelle_metadata = {
      "name": "Mouse Nucleus Accumbens Dataset",
      "dataset_id": "jrc_mus-nacc-2",
      "source": "Janelia / COSEM",
      "organism": "mouse",
      "strain": "C57/BL6J",
      "sex": "male",
      "age": "adult",
      "brain_region": "nucleus accumbens",
      "modality": "FIB-SEM (Focused Ion Beam Scanning Electron Microscopy)",
      "sample_preparation": {
        "fixation": "chemical fixation",
        "staining": [
          "reduced osmium processing",
          "2% samarium trichloride",
          "1% uranyl acetate in maleate buffer"
        ],
        "embedding": "Durcupan resin"
      },
      "acquisition": {
        "imaging_start_date": "2015-03-09",
        "imaging_duration_days": 1,
        "voxel_resolution_nm": [4, 4, 2.96],
        "volume_dimensions_nm": [10384, 10080, 1669.44],
        "fib_sem_parameters": {
          "bias_volts": 500,
          "scan_rate_mhz": 0.4,
          "current_na": 0.22,
          "primary_energy_ev": 200
        }
      },
      "data_access": {
        "storage_location": "s3://janelia-cosem-datasets/jrc_mus-nacc-2/",
        "modalities_available": [
          "EM data",
          "FIB-SEM data"
        ]
      },
      "contributors": [
        "Richard Weinberg (UNC)",
        "C. Shan Xu (Yale)",
        "Kenneth Hayworth (HHMI/Janelia)",
        "Gleb Shtengel (HHMI/Janelia)",
        "Harald Hess (HHMI/Janelia)"
      ],
      "doi": {
        "fib_sem_reconstruction": "10.25378/janelia.24222898",
        "nuclei_segmentations": "10.25378/janelia.26513209"
      },
      "publications": [
        {
          "authors": "Xu et al.",
          "year": 2021,
          "doi": "10.1038/s41586-021-03992-4"
        },
        {
          "authors": "Xu et al.",
          "year": 2017,
          "doi": "10.7554/eLife.25916"
        },
        {
          "authors": "Wu et al.",
          "year": 2017,
          "doi": "10.1073/pnas.1701078114"
        }
      ]
    }

In [6]:
# Consolidated metadata schema - normalize all five datasets
import pandas as pd
def consolidate_metadata(empiar_metadata, hemibrain_metadata, idr_metadata, epfl_metadata, openorganelle_metadata):

    # --- Helpers ---
    def empiar_resolution_nm():
        """EMPIAR pixel_width/height are in Angstroms, z is in the details text."""
        xy = empiar_metadata["EMPIAR-11759"]["imagesets"][0].get("pixel_width")
        xy_nm = xy / 10.0 if xy else None  # Angstroms -> nm
        # z from details: "50nm per section"
        details = empiar_metadata["EMPIAR-11759"]["imagesets"][0].get("details", "")
        z_nm = 50.0  # parsed from details string
        return [xy_nm, xy_nm, z_nm] if xy_nm else None

    def empiar_volume_voxels():
        imgset = empiar_metadata["EMPIAR-11759"]["imagesets"][0]
        w = int(imgset.get("image_width", 0))
        h = int(imgset.get("image_height", 0))
        z = imgset.get("num_images_or_tilt_series", 0)
        return [w, h, z] if w and h and z else None

    def idr_resolution_nm():
        ps = idr_metadata.get("pixel_size", {})
        # IDR pixel_size is in micrometers
        return [ps["x"] * 1000, ps["y"] * 1000, ps["z"] * 1000]

    def idr_volume_voxels():
        s = idr_metadata.get("size", {})
        return [s.get("width"), s.get("height"), s.get("z")]

    def compute_physical_volume(voxels, res_nm):
        if voxels and res_nm and all(v and r for v, r in zip(voxels, res_nm)):
            return [v * r for v, r in zip(voxels, res_nm)]
        return None

    # --- Build each entry ---

    hemibrain_res = [float(r) for r in hemibrain_metadata.get("resolution_nm", [])]
    hemibrain_shape = [int(s) for s in hemibrain_metadata.get("shape", [])]

    datasets = [
        {
            "dataset_id": "jrc_mus-nacc-2",
            "name": openorganelle_metadata["name"],
            "source": openorganelle_metadata["source"],
            "doi": openorganelle_metadata.get("doi", {}).get("fib_sem_reconstruction"),
            "organism": openorganelle_metadata.get("organism"),
            "strain_or_cell_line": openorganelle_metadata.get("strain"),
            "sex": openorganelle_metadata.get("sex"),
            "age": openorganelle_metadata.get("age"),
            "tissue_or_region": openorganelle_metadata.get("brain_region"),
            "modality": openorganelle_metadata.get("modality"),
            "voxel_resolution_nm": openorganelle_metadata["acquisition"]["voxel_resolution_nm"],
            "volume_dimensions_voxels": None,  # not directly provided
            "volume_dimensions_nm": openorganelle_metadata["acquisition"]["volume_dimensions_nm"],
            "dtype": None,
            "num_channels": None,
            "file_format": None,
            "acquisition_params": openorganelle_metadata["acquisition"].get("fib_sem_parameters"),
            "annotations_available": None,
            "contributors": openorganelle_metadata.get("contributors"),
            "publications": [
                {"authors": p["authors"], "year": p.get("year"), "doi": p.get("doi")}
                for p in openorganelle_metadata.get("publications", [])
            ],
            "data_url": openorganelle_metadata.get("data_access", {}).get("storage_location"),
            "deposition_date": None,
            "release_date": None,
        },
        {
            "dataset_id": "epfl_hippocampus_em",
            "name": epfl_metadata["name"],
            "source": epfl_metadata["source"],
            "doi": None,
            "organism": epfl_metadata.get("organism"),
            "strain_or_cell_line": None,
            "sex": None,
            "age": None,
            "tissue_or_region": epfl_metadata.get("brain_region"),
            "modality": epfl_metadata.get("modality"),
            "voxel_resolution_nm": epfl_metadata.get("voxel_resolution_nm"),
            "volume_dimensions_voxels": epfl_metadata.get("volume_size_voxels"),
            "volume_dimensions_nm": compute_physical_volume(
                epfl_metadata.get("volume_size_voxels"),
                epfl_metadata.get("voxel_resolution_nm"),
            ),
            "dtype": None,
            "num_channels": None,
            "file_format": epfl_metadata.get("file_format"),
            "acquisition_params": None,
            "annotations_available": epfl_metadata.get("annotation", {}).get("annotated_structures"),
            "contributors": epfl_metadata.get("acquisition", {}).get("contributors"),
            "publications": [
                {
                    "authors": None,
                    "year": None,
                    "doi": None,
                    "title": epfl_metadata.get("related_publication", {}).get("title"),
                    "venue": epfl_metadata.get("related_publication", {}).get("venue"),
                }
            ],
            "data_url": None,
            "deposition_date": None,
            "release_date": None,
        },
        {
            "dataset_id": "EMPIAR-11759",
            "name": empiar_metadata["EMPIAR-11759"]["title"],
            "source": "EMPIAR",
            "doi": empiar_metadata["EMPIAR-11759"].get("entry_doi"),
            "organism": "zebrafish",
            "strain_or_cell_line": None,
            "sex": None,
            "age": "55 hpf larva",
            "tissue_or_region": "developing retina",
            "modality": empiar_metadata["EMPIAR-11759"].get("experiment_type"),
            "voxel_resolution_nm": empiar_resolution_nm(),
            "volume_dimensions_voxels": empiar_volume_voxels(),
            "volume_dimensions_nm": compute_physical_volume(
                empiar_volume_voxels(), empiar_resolution_nm()
            ),
            "dtype": empiar_metadata["EMPIAR-11759"]["imagesets"][0].get("voxel_type"),
            "num_channels": None,
            "file_format": empiar_metadata["EMPIAR-11759"]["imagesets"][0].get("data_format"),
            "acquisition_params": {
                "voltage_kv": 1.9,
                "current_pa": 200,
                "dwell_time_us": 0.5,
            },
            "annotations_available": None,
            "contributors": [
                f"{a['author']['name']}"
                for a in empiar_metadata["EMPIAR-11759"].get("authors", [])
            ],
            "publications": [
                {
                    "authors": c.get("authors", [{}])[0].get("name") if c.get("authors") else None,
                    "year": c.get("year"),
                    "doi": c.get("doi"),
                    "title": c.get("title"),
                }
                for c in empiar_metadata["EMPIAR-11759"].get("citation", [])
            ],
            "data_url": None,
            "deposition_date": empiar_metadata["EMPIAR-11759"].get("deposition_date"),
            "release_date": empiar_metadata["EMPIAR-11759"].get("release_date"),
        },
        {
            "dataset_id": "idr0086-U2OS-chromatin",
            "name": idr_metadata["meta"]["imageName"],
            "source": "IDR (Image Data Resource)",
            "doi": None,
            "organism": "human (cell line)",
            "strain_or_cell_line": "U2OS",
            "sex": None,
            "age": None,
            "tissue_or_region": "nucleus (chromatin)",
            "modality": "FIB-SEM",
            "voxel_resolution_nm": idr_resolution_nm(),
            "volume_dimensions_voxels": idr_volume_voxels(),
            "volume_dimensions_nm": compute_physical_volume(
                idr_volume_voxels(), idr_resolution_nm()
            ),
            "dtype": idr_metadata["meta"].get("pixelsType"),
            "num_channels": idr_metadata["size"].get("c"),
            "file_format": "TIFF",
            "acquisition_params": None,
            "annotations_available": None,
            "contributors": [idr_metadata["meta"].get("imageAuthor", "")],
            "publications": [
                {
                    "authors": None,
                    "year": None,
                    "doi": None,
                    "title": "Chromatin arranges in chains of mesoscale domains with nanoscale functional topography independent of cohesin",
                }
            ],
            "data_url": None,
            "deposition_date": None,
            "release_date": None,
        },
        {
            "dataset_id": "janelia_hemibrain_v1.2",
            "name": "Janelia FlyEM Hemibrain",
            "source": "Janelia FlyEM / Neuroglancer",
            "doi": None,
            "organism": "Drosophila melanogaster",
            "strain_or_cell_line": None,
            "sex": None,
            "age": None,
            "tissue_or_region": "hemibrain",
            "modality": "FIB-SEM",
            "voxel_resolution_nm": hemibrain_res,
            "volume_dimensions_voxels": hemibrain_shape,
            "volume_dimensions_nm": compute_physical_volume(hemibrain_shape, hemibrain_res),
            "dtype": hemibrain_metadata.get("dtype"),
            "num_channels": hemibrain_metadata.get("info", {}).get("num_channels"),
            "file_format": "Neuroglancer precomputed (sharded, JPEG)",
            "acquisition_params": None,
            "annotations_available": None,
            "contributors": None,
            "publications": None,
            "data_url": hemibrain_metadata.get("source"),
            "deposition_date": None,
            "release_date": None,
        },
    ]

    return datasets


# --- Run it ---
consolidated = consolidate_metadata(
    empiar_metadata, hemibrain_metadata, idr_metadata, epfl_metadata, openorganelle_metadata
)
df = pd.DataFrame(consolidated)
df.to_csv(DATA_DIR / "consolidated_metadata.csv", index=False)
# print(df.to_string())
df


# Quick inspection
# for ds in consolidated:
#     print(f"\n{'='*60}")
#     print(f"  {ds['dataset_id']}  —  {ds['name']}")
#     print(f"  Source:      {ds['source']}")
#     print(f"  Organism:    {ds['organism']}")
#     print(f"  Modality:    {ds['modality']}")
#     print(f"  Resolution:  {ds['voxel_resolution_nm']} nm")
#     print(f"  Volume (vx): {ds['volume_dimensions_voxels']}")
#     print(f"  dtype:       {ds['dtype']}")

,dataset_id,name,source,doi,organism,strain_or_cell_line,sex,age,tissue_or_region,modality,...,dtype,num_channels,file_format,acquisition_params,annotations_available,contributors,publications,data_url,deposition_date,release_date
0,jrc_mus-nacc-2,Mouse Nucleus Accumbens Dataset,Janelia / COSEM,10.25378/janelia.24222898,mouse,C57/BL6J,male,adult,nucleus accumbens,FIB-SEM (Focused Ion Beam Scanning Electron Mi...,...,None,NaN,None,"{'bias_volts': 500, 'scan_rate_mhz': 0.4, 'cur...",None,"[Richard Weinberg (UNC), C. Shan Xu (Yale), Ke...","[{'authors': 'Xu et al.', 'year': 2021, 'doi':...",s3://janelia-cosem-datasets/jrc_mus-nacc-2/,None,None
1,epfl_hippocampus_em,EPFL Electron Microscopy Dataset,EPFL,None,unknown (brain tissue),None,None,None,CA1 hippocampus,Electron Microscopy,...,None,NaN,multipage TIFF,None,[mitochondria],"[Graham Knott, Marco Cantoni]","[{'authors': None, 'year': None, 'doi': None, ...",None,None,None
2,EMPIAR-11759,Developing retina in zebrafish 55 hpf larval eye.,EMPIAR,10.6019/EMPIAR-11759,zebrafish,None,None,55 hpf larva,developing retina,SBF-SEM,...,UNSIGNED BYTE,NaN,DM3,"{'voltage_kv': 1.9, 'current_pa': 200, 'dwell_...",None,[Wilsch-Bräuninger M],"[{'authors': 'Wilsch-Bräuninger M', 'year': No...",None,2023-11-01,2024-01-15
3,idr0086-U2OS-chromatin,Figure_S3B_FIB-SEM_U2OS_20x20x20nm_xy.tif,IDR (Image Data Resource),None,human (cell line),U2OS,None,None,nucleus (chromatin),FIB-SEM,...,uint8,1.0,TIFF,None,None,[Public data],"[{'authors': None, 'year': None, 'doi': None, ...",None,None,None
4,janelia_hemibrain_v1.2,Janelia FlyEM Hemibrain,Janelia FlyEM / Neuroglancer,None,Drosophila melanogaster,None,None,None,hemibrain,FIB-SEM,...,uint8,1.0,"Neuroglancer precomputed (sharded, JPEG)",None,None,None,None,precomputed://gs://neuroglancer-janelia-flyem-...,None,None


## How the consolidated metadata table was created

### Data collection
Metadata was gathered from two types of sources:
- **Local metadata files**: For three datasets (EMPIAR-11759, Hemibrain, IDR), metadata JSON 
  files were already available in the `data/` directory, either downloaded alongside the image 
  data or fetched from the respective web APIs (e.g., OMERO's `imgData` endpoint for IDR, 
  EMPIAR's REST API).
- **Manual curation from web pages**: For the remaining two datasets (EPFL and OpenOrganelle), 
  no structured metadata file was bundled with the data. Metadata was manually extracted from 
  the dataset landing pages and documentation, then stored as Python dictionaries.

### Consolidation process
Each dataset stores metadata in a different schema with different field names, nesting, and 
units. To produce a single unified table, I:

1. **Identified common fields** across all five datasets by inspecting each metadata dictionary 
   and mapping semantically equivalent keys (e.g., `pixel_size` in IDR vs `voxel_resolution_nm` 
   in EPFL vs `resolution_nm` in Hemibrain all represent the same concept).
2. **Normalised units** to a consistent standard:
   - All resolutions were converted to **nanometers** (EMPIAR `pixel_width`/`pixel_height` were 
     in Ångströms ÷ 10; IDR `pixel_size` was in micrometers × 1000).
   - Volume dimensions are reported in both voxels and nanometers where computable.
3. **Filled in contextual metadata** that was not explicitly in the structured files but is 
   known from the dataset documentation (e.g., Hemibrain organism = *Drosophila melanogaster*, 
   EMPIAR tissue = developing retina, EMPIAR z-resolution = 50 nm from the free-text description 
   field).
4. **Kept all fields nullable**: since datasets vary in how much metadata they expose, fields 
   that are unavailable for a given dataset are set to `None` rather than omitted.

### Schema
The consolidated table uses one row per dataset with the following columns: `dataset_id`, 
`name`, `source`, `doi`, `organism`, `strain_or_cell_line`, `sex`, `age`, `tissue_or_region`, 
`modality`, `voxel_resolution_nm` (x, y, z in nm), `volume_dimensions_voxels`, 
`volume_dimensions_nm`, `dtype`, `num_channels`, `file_format`, `acquisition_params`, 
`annotations_available`, `contributors`, `publications`, `data_url`, `deposition_date`, 
`release_date`.

The consolidation code and an LLM (Claude) were used to help identify corresponding fields 
across the heterogeneous schemas and to generate the mapping/conversion code.